# 500M Transformer Training on Single v5e TPU
Single-chip v5e, full BF16 precision, indefinite training with Google Drive checkpointing

## Setup

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/gdrive')
print("Google Drive mounted!")

In [ ]:
# Install JAX for TPU
!pip install -q jax[tpu] -f https://storage.googleapis.com/jax-releases/libtpu_releases.html

In [ ]:
# Install dependencies
!pip install -q transformers datasets tiktoken pyyaml tqdm numpy

In [ ]:
# Clone or upload dLLM repo
import os
os.chdir('/content')

if not os.path.exists('dLLM'):
    !git clone https://github.com/Z-Coder672/dLLM.git
else:
    print("dLLM repo already present")

os.chdir('/content/dLLM')
!pwd

In [ ]:
# Verify TPU setup
import jax

devices = jax.devices()
print(f"\n=== TPU Setup ===")
print(f"Available devices: {len(devices)}")
for i, device in enumerate(devices):
    print(f"  Device {i}: {device.device_kind}")

print(f"\nJAX version: {jax.__version__}")
print(f"Default device: {jax.default_device()}")

## Training Configuration

In [ ]:
# Display v5e config
import yaml

with open('/content/dLLM/configs/v5e.yaml', 'r') as f:
    config = yaml.safe_load(f)

print("\n=== Model Config ===")
for k, v in config['model'].items():
    print(f"  {k}: {v}")

print("\n=== Training Config ===")
for k, v in config['training'].items():
    if k not in ['datasets']:
        print(f"  {k}: {v}")
    else:
        print(f"  datasets:")
        for ds in v:
            print(f"    - {ds['name']}: weight={ds.get('weight', 1.0)}")

## Start Training

In [ ]:
# Check checkpoint directory
from pathlib import Path
import json

ckpt_dir = '/content/gdrive/MyDrive/checkpoints/500m_v5e'
ckpt_path = Path(ckpt_dir)

if ckpt_path.exists():
    checkpoints = sorted([d for d in ckpt_path.iterdir() if d.is_dir()], 
                         key=lambda x: int(x.name.split('_')[-1]) if 'step_' in x.name else 0)
    if checkpoints:
        latest = checkpoints[-1]
        state_file = latest / 'state.json'
        if state_file.exists():
            with open(state_file) as f:
                state = json.load(f)
            print(f"Latest checkpoint: {latest.name}")
            print(f"  Step: {state.get('step', 0)}")
            print(f"\nResume with: python train_v5e_complete.py --config configs/v5e.yaml --resume {latest}")
else:
    print(f"Checkpoint directory will be created at: {ckpt_dir}")
    Path(ckpt_dir).mkdir(parents=True, exist_ok=True)

In [ ]:
# Start training (fresh or resume)
import subprocess

# For fresh start:
cmd = ['python', 'train_v5e_complete.py', '--config', 'configs/v5e.yaml']

# For resume (uncomment and fill in path):
# cmd = ['python', 'train_v5e_complete.py', '--config', 'configs/v5e.yaml', 
#        '--resume', '/content/gdrive/MyDrive/checkpoints/500m_v5e/step_1000']

print(f"Running: {' '.join(cmd)}")
subprocess.run(cmd, cwd='/content/dLLM')

## Monitoring

In [ ]:
# Check training progress
import json
from pathlib import Path

ckpt_dir = Path('/content/gdrive/MyDrive/checkpoints/500m_v5e')

checkpoints = sorted(
    [d for d in ckpt_dir.iterdir() if d.is_dir() and (d / 'state.json').exists()],
    key=lambda x: int(x.name.split('_')[-1]) if 'step_' in x.name else 0
)

if checkpoints:
    print("Recent checkpoints:")
    for ckpt in checkpoints[-5:]:
        state_file = ckpt / 'state.json'
        with open(state_file) as f:
            state = json.load(f)
        print(f"  {ckpt.name}: step={state.get('step', 0)}")
else:
    print("No checkpoints yet")

In [ ]:
# View training log
log_file = '/content/dLLM/training_v5e_complete.log'
if os.path.exists(log_file):
    with open(log_file, 'r') as f:
        lines = f.readlines()
    
    # Show last 30 lines
    print(f"Last 30 lines of training log:")
    print(''.join(lines[-30:]))
else:
    print("Training log not found")